# Compare / Review — Iteration Notebook

Polish how the General Specialist detects contradictions and cross-domain insights.

**Edit surfaces:**
- Non-engineer: `skills/workflow/comparison.md`.
- Engineer: `agents/general_specialist.py`.

**Regeneration:** set `REGENERATE=True` and Cell 4 runs a live team dispatch to capture fresh `specialist_outputs` from the configured case; otherwise it reads the hand-authored fixture JSON.

**Workflow:** edit a file above → Run All → read Cell 5's rendered ReviewReport → repeat. Cell 6's raw JSON is the regression signal.

In [ ]:
# ═══════════════ KNOBS ═══════════════
FIXTURE = "basic_case"
REGENERATE = False          # True = run live team dispatch and snapshot specialist_outputs
MODEL = "gpt-4.1"
# ═════════════════════════════════════

import json
import os
import sys
from pathlib import Path

import nest_asyncio
nest_asyncio.apply()

from dotenv import load_dotenv
load_dotenv()

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from agents.general_specialist import GeneralSpecialist
from agents.session_registry import SessionRegistry
from config.pillar_loader import PillarLoader
from datalayer.catalog import DataCatalog
from datalayer.gateway import SimulatedDataGateway
from datalayer.generator import DataGenerator
from gateway.firewall_stack import FirewallStack
from gateway.llm_factory import build_llm
from logger.event_logger import EventLogger
from models.types import SpecialistOutput
from orchestrator.orchestrator import Orchestrator
from skills.domain.loader import list_domain_skills, load_domain_skill
from tools.data_tools import init_tools

FIXTURE_PATH = PROJECT_ROOT / "notebooks" / "fixtures" / "compare_review" / f"{FIXTURE}.json"
print(f"Fixture: {FIXTURE_PATH.relative_to(PROJECT_ROOT)}")

In [ ]:
logger = EventLogger(session_id="polish-compare-review")
firewall = FirewallStack(logger=logger)
llm = build_llm(MODEL, firewall)

_DATA_TABLES_DIR = PROJECT_ROOT / "data_tables"
csv_gateway = SimulatedDataGateway.from_case_folders(str(_DATA_TABLES_DIR))
if csv_gateway.list_case_ids():
    gateway = csv_gateway
else:
    gen = DataGenerator(
        profile_dir=str(PROJECT_ROOT / "config" / "data_profiles"),
        seed=42, cases=50,
    )
    gen.load_profiles()
    tables_raw = gen.generate_all()
    gateway = SimulatedDataGateway.from_generated(tables_raw)

catalog = DataCatalog(profile_dir=str(PROJECT_ROOT / "config" / "data_profiles"))
init_tools(gateway, catalog, logger=logger)
registry = SessionRegistry()
pillar_loader = PillarLoader(pillar_dir=str(PROJECT_ROOT / "config" / "pillars"))
print(f"Available case IDs (first 5): {gateway.list_case_ids()[:5]}")

In [ ]:
if REGENERATE:
    # Live partial team run: dispatch each specialist against a real case and
    # snapshot their outputs. Uses the orchestrator's own planning path so
    # the captured outputs match what the full pipeline would produce.
    REGEN_PILLAR = "credit_risk"
    REGEN_CASE = gateway.list_case_ids()[0]
    REGEN_QUESTION = "Is this customer's credit risk acceptable?"

    gateway.set_case(REGEN_CASE)
    pillar_yaml = pillar_loader.load(REGEN_PILLAR) or {}
    orchestrator = Orchestrator(
        llm, logger, registry, REGEN_PILLAR,
        pillar_config=pillar_yaml, catalog=catalog,
    )
    plan = await orchestrator.plan_team(
        question=REGEN_QUESTION,
        available_specialists=list_domain_skills(),
        active_specialists=[],
    )

    import asyncio
    async def _dispatch(assignment):
        skill = load_domain_skill(assignment.specialist)
        if skill is None:
            return None
        agent = registry.get_or_create(
            domain=assignment.specialist,
            pillar=REGEN_PILLAR,
            domain_skill=skill,
            pillar_yaml=pillar_yaml,
            llm=llm,
            logger=logger,
        )
        out = await agent.run(assignment.sub_question, mode="chat", root_question=REGEN_QUESTION)
        return assignment.specialist, out

    pairs = await asyncio.gather(*(_dispatch(a) for a in plan))
    specialist_outputs = {name: out for pair in pairs if pair is not None for name, out in [pair]}

    current = {
        "question": REGEN_QUESTION,
        "specialist_outputs": {d: o.model_dump() for d, o in specialist_outputs.items()},
        "notes": f"Regenerated from case {REGEN_CASE}, pillar {REGEN_PILLAR}.",
    }
    FIXTURE_PATH.write_text(json.dumps(current, indent=2) + "\n")
    fixture = current
    print(f"Wrote fixture with {len(specialist_outputs)} specialist outputs: {FIXTURE_PATH.relative_to(PROJECT_ROOT)}")
else:
    fixture = json.loads(FIXTURE_PATH.read_text())
    specialist_outputs = {
        domain: SpecialistOutput(**payload)
        for domain, payload in fixture["specialist_outputs"].items()
    }
    print(f"Loaded fixture: {FIXTURE_PATH.relative_to(PROJECT_ROOT)}")

print(f"Question: {fixture['question']}")
print(f"Specialist outputs: {list(specialist_outputs.keys())}")

In [ ]:
from IPython.display import Markdown, display

general = GeneralSpecialist(llm, logger)
review = await general.compare(specialist_outputs, fixture["question"])

lines = [f"### Question\n\n{fixture['question']}\n"]
lines.append(f"**Specialists compared:** {', '.join(specialist_outputs.keys())}\n")

lines.append(f"**Resolved contradictions ({len(review.resolved)}):**")
if not review.resolved:
    lines.append("- _(none)_")
for r in review.resolved:
    lines.append(f"- **{r.pair[0]} ↔ {r.pair[1]}** — {r.contradiction}")
    lines.append(f"  - Q: _{r.question_raised}_")
    lines.append(f"  - A: {r.answer}")
    lines.append(f"  - Conclusion: {r.conclusion}")

lines.append(f"\n**Open conflicts ({len(review.open_conflicts)}):**")
if not review.open_conflicts:
    lines.append("- _(none)_")
for c in review.open_conflicts:
    lines.append(f"- **{c.pair[0]} ↔ {c.pair[1]}** — {c.contradiction}")
    lines.append(f"  - Unresolved because: {c.reason_unresolved}")

lines.append(f"\n**Cross-domain insights ({len(review.cross_domain_insights)}):**")
if not review.cross_domain_insights:
    lines.append("- _(none)_")
for ins in review.cross_domain_insights:
    lines.append(f"- {ins}")

display(Markdown("\n".join(lines)))

In [ ]:
print(json.dumps(review.model_dump(), indent=2))